In [ ]:
import cv2
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np

def parse_head_boxes(xml_path):
    # Extract head bounding boxes from CVAT XML.
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    head_boxes = {}
    for track in root.findall('.//track[@label="head"]'):
        for box in track.findall('box'):
            frame_id = int(box.get('frame'))
            head_boxes[frame_id] = {
                'xtl': float(box.get('xtl')),
                'ytl': float(box.get('ytl')),
                'xbr': float(box.get('xbr')),
                'ybr': float(box.get('ybr'))
            }
    return head_boxes


def blur_faces(images_dir, annotations_xml, output_dir, blur_strength=99, shrink_factor=1.0):
    # Blur faces with circular mask
    images_dir = Path(images_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    head_boxes = parse_head_boxes(annotations_xml)
    image_files = sorted(images_dir.glob('frame_*.PNG'))
    
    # Define skip range
    skip_start = 96
    skip_end = 119
    
    for idx, img_path in enumerate(image_files):
        img = cv2.imread(str(img_path))
        
        # Check if frame should be skipped (only skip blurring, not saving)
        should_blur = not (skip_start <= idx <= skip_end)
        
        if idx in head_boxes and should_blur:
            box = head_boxes[idx]
            
            # Calculate center and dimensions
            x1, y1 = int(box['xtl']), int(box['ytl'])
            x2, y2 = int(box['xbr']), int(box['ybr'])
            
            # Shrink the box
            width = x2 - x1
            height = y2 - y1
            shrink_x = int(width * (1 - shrink_factor) / 2)
            shrink_y = int(height * (1 - shrink_factor) / 2)
            
            x1 += shrink_x
            x2 -= shrink_x
            y1 += shrink_y
            y2 -= shrink_y
            
            # Calculate center and radius for ellipse
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            radius_x = (x2 - x1) // 2
            radius_y = (y2 - y1) // 2
            
            # Create circular/elliptical mask
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
            cv2.ellipse(mask, (center_x, center_y), (radius_x, radius_y), 0, 0, 360, 255, -1)
            
            # Blur entire image
            blurred_img = cv2.GaussianBlur(img, (blur_strength, blur_strength), 0)
            
            # Use mask to blend original and blurred
            mask_3channel = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
            img = np.where(mask_3channel == 255, blurred_img, img)
        
        output_path = output_dir / img_path.name
        cv2.imwrite(str(output_path), img)
        
        skip_msg = " (skipped blur - facing away)" if not should_blur else ""
        print(f"Processed {img_path.name}{skip_msg}")

# Usage
blur_faces(
    images_dir='./data/annotations/images/default',
    annotations_xml='./data/annotations/annotations.xml',
    output_dir='./data/annotations/images/default'
)

Processed frame_000000.PNG
Processed frame_000001.PNG
Processed frame_000002.PNG
Processed frame_000003.PNG
Processed frame_000004.PNG
Processed frame_000005.PNG
Processed frame_000006.PNG
Processed frame_000007.PNG
Processed frame_000008.PNG
Processed frame_000009.PNG
Processed frame_000010.PNG
Processed frame_000011.PNG
Processed frame_000012.PNG
Processed frame_000013.PNG
Processed frame_000014.PNG
Processed frame_000015.PNG
Processed frame_000016.PNG
Processed frame_000017.PNG
Processed frame_000018.PNG
Processed frame_000019.PNG
Processed frame_000020.PNG
Processed frame_000021.PNG
Processed frame_000022.PNG
Processed frame_000023.PNG
Processed frame_000024.PNG
Processed frame_000025.PNG
Processed frame_000026.PNG
Processed frame_000027.PNG
Processed frame_000028.PNG
Processed frame_000029.PNG
Processed frame_000030.PNG
Processed frame_000031.PNG
Processed frame_000032.PNG
Processed frame_000033.PNG
Processed frame_000034.PNG
Processed frame_000035.PNG
Processed frame_000036.PNG
P